In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("fullGas.csv")
df.columns

ModuleNotFoundError: No module named 'pandas'

In [ ]:
display(df.shape) #Our initial datafram has 40494 rows and 25 columns
df.isna().sum() # Model, Body, Year, Country, Fuel_Type, Fu
na_cols = [x for x in df.columns if df[x].isna().sum() != 0 ]
na_cols
df[df['Gears'].isnull()]
# na_cols ['Model','Body','Year','Country','Fuel_Type','Fuel_Consumption_l','Drivetrain','Gearbox','Gears',\
#          'Engine_Size_cc','Cylinders','Seats','Doors','Color','Upholstery','Previous_Owners','Seller']

na_dict = {x: df[x].isnull().sum() for x in df.columns if df[x].isnull().sum != 0}
na_dict
max(na_dict, key=lambda x:x)
df[df['Gears'].isnull() == df['Year'].isnull()]

sorted_na = pd.Series(na_dict, index=na_cols).sort_values(ascending=False).to_dict()
sorted_na
df[df['Mileage_km'] == 0]['Mileage_km']
df[df['Year'].isna()]['Previous_Owners']
df[df['Gears'].isna()].loc[: , 'Make':'Gears']
# df[df['Gearbox'].isna()].loc[:, ['Gearbox','Gears']].corr()
na_dict2 = na_dict.copy()
for key, val in na_dict2.items():
  if val >= df.shape[0] * 0.3:
    del na_dict[key]

na_dict




(40494, 25)

{'Make': np.int64(0),
 'Model': np.int64(221),
 'Body': np.int64(1),
 'Mileage_km': np.int64(0),
 'Price': np.int64(0),
 'Year': np.int64(1695),
 'Country': np.int64(2592),
 'Condition': np.int64(0),
 'Fuel_Type': np.int64(29),
 'Drivetrain': np.int64(8634),
 'Gearbox': np.int64(480),
 'Power_hp': np.int64(0),
 'Engine_Size_cc': np.int64(5930),
 'Cylinders': np.int64(10162),
 'Seats': np.int64(1797),
 'Doors': np.int64(977),
 'Color': np.int64(2575),
 'Upholstery': np.int64(7272),
 'Full_Service_History': np.int64(0),
 'Non_Smoker_Vehicle': np.int64(0),
 'Seller': np.int64(150),
 'Image_url': np.int64(0)}

In [ ]:
import seaborn as sns
# If 20% or more of a feature's values is NaN, the feature will be dropped. No serious training can be done on
# such features
to_drop = df[[x for x in df.columns if df[x].isna().sum() >= (df.shape[0] * 0.2)]].columns.tolist() # These will be dropped
to_fix = df[[x for x in df.columns if df[x].isna().sum() < (df.shape[0] * 0.2) and df[x].isna().sum() > 0]].columns  # These will be fixed
print(to_fix)
df_floats = df[to_fix].select_dtypes(include='float64')
df_objects = df[to_fix].select_dtypes(include='object')
#df[df[to_fix].dtypes == np.float64]
#floats = ['Year', 'Engine_Size_cc', 'Seats', 'Doors']
df_objects[df_objects['Model'].isna()]
df_objects['Model'].isna().sum() #221
df_objects['Country'].isna().sum() #2592
df_objects[df_objects['Model'].isna() & df_objects['Country'].isna()] #34
df_objects[df_objects['Model'].isna() & df_objects['Upholstery'].isna()].shape #56
df_objects.isna().corr()
df_floats



Index(['Model', 'Body', 'Year', 'Country', 'Fuel_Type', 'Gearbox',
       'Engine_Size_cc', 'Seats', 'Doors', 'Color', 'Upholstery', 'Seller'],
      dtype='object')


,Year,Engine_Size_cc,Seats,Doors
0,2020.0,1368.0,4.0,3.0
1,2017.0,1368.0,4.0,3.0
2,2015.0,1368.0,4.0,3.0
3,2008.0,1368.0,4.0,3.0
4,2021.0,1368.0,4.0,3.0
...,...,...,...,...
40489,2023.0,1969.0,5.0,5.0
40490,2020.0,1969.0,5.0,5.0
40491,2020.0,1969.0,5.0,5.0
40492,2022.0,1477.0,5.0,5.0


In [ ]:
to_fix2 = df[[x for x in df.columns if df[x].isna().sum() < (df.shape[0] * 0.2) and df[x].isna().sum() >= 0]].columns  # These will be fixed
print(to_fix2)
print(to_fix)

np.isin(to_fix, to_fix2).all()

Index(['Make', 'Model', 'Body', 'Mileage_km', 'Price', 'Year', 'Country',
       'Condition', 'Fuel_Type', 'Gearbox', 'Power_hp', 'Engine_Size_cc',
       'Seats', 'Doors', 'Color', 'Upholstery', 'Full_Service_History',
       'Non_Smoker_Vehicle', 'Seller', 'Image_url'],
      dtype='object')
Index(['Model', 'Body', 'Year', 'Country', 'Fuel_Type', 'Gearbox',
       'Engine_Size_cc', 'Seats', 'Doors', 'Color', 'Upholstery', 'Seller'],
      dtype='object')


np.True_

In [ ]:

df_objects.describe()
#df_objects['Model'].isna().sum() # 221 nan values in Model
df_reset = df_objects['Model'].value_counts().reset_index()
df_reset['count'].median() * 628 #5024
below_50_percent = df_reset[df_reset['count'] <= 8]['Model'].values
#df_objects.index[df_objects['Model'] == below_50_percent[i] for i in range(len(below_50_percent) + 1)]
# temp = [(df_objects['Model'] == below_50_percent[i]).index.tolist() for i in range(len(below_50_percent))]
# temp



In [ ]:
from collections import defaultdict
df_objects[df_objects['Country'].isna() & df_objects['Color'].isna()] # 996 rows
df_objects[df_objects['Country'].isna() & df_objects['Color'].isna() & df_objects['Upholstery'].isna()] #950
df_objects.describe()
df_objects['Country'].unique() #['IT', 'BE', 'DE', 'NL', 'AT', 'LU', 'ES', 'FR', nan]
df_objects['Upholstery'].unique() #['Full leather', 'Cloth', 'Part leather', 'alcantara', nan, 'Other','Velour']
df_objects['Color'].unique() #['White', 'Grey', 'Bronze', 'Black', 'Yellow', nan, 'Blue', 'Green','Violet', 'Red', 'Orange', 'Silver', 'Brown', 'Beige', 'Gold']

#df_objects['Body'].value_counts(dropna=False).plot(kind='bar')# we can drop the na row for this feature - 1
#df_objects['Fuel_Type'].isna().sum() # we can drop the na rows for this feature - 29
#df_objects['Seller'].isna().sum() # we can drop the na rows for this feature-150
#df_objects['Gearbox'].isna().sum() # we can drop the na rows for this feature-480

# We are dropping a total of 660 rows from our categorical features. This amounts to ~ 1.6% of the objects df df_objects

df_objects['Country_impu'] = df_objects['Country'].replace(np.nan, df_objects['Country'].value_counts().index[0])
df_objects['Color_impu'] = df_objects['Color'].replace(np.nan, df_objects['Color'].value_counts().index[0])
df_objects['Upholstery_impu'] = df_objects['Upholstery'].replace(np.nan, df_objects['Upholstery'].value_counts().index[0])

a = df[df_objects['Body'].isna()].index.tolist()
b = df[df_objects['Fuel_Type'].isna()].index.tolist()
c = df[df_objects['Seller'].isna()].index.tolist()
d = df[df_objects['Gearbox'].isna()].index.tolist()
ind_drop = a+b+c+d
df_objects.columns

ind_drop2 = np.concatenate((np.where([df_objects['Model'].isna(), df_objects['Gearbox'].isna(), df_objects['Fuel_Type'].isna()])[1], np.where([df_objects['Body'].isna(), df_objects['Seller'].isna()])[1]), axis=0)
ind_drop

tally = defaultdict(list)

for i,v in enumerate(ind_drop):
  tally[v].append(i)
tally








In [ ]:
# Drop columns Country, Color, and Upholstery since we now have their imputed versions
# Also drop the .isna() rows corresponding to columns Body, Fuel_Type, Seller, and Gearbox since they are so few
Xcat = df.select_dtypes(include='object').drop(['Country','Color','Upholstery', 'Image_url'], axis=1)
# Xcat.info()
Xcat.isna().sum()
Xcat['Drivetrain'].unique() #['Front Wheel Drive', nan, 'Rear Wheel Drive', '4WD']
# plt.hist(Xcat[Xcat['Drivetrain'] == 'Front Wheel Drive'].index, bins=30, alpha=0.5, label='Front Wheel')
# plt.hist(Xcat[Xcat['Drivetrain'] == 'Rear Wheel Drive'].index, bins=30, alpha=0.9, label='Rear Wheel')
# plt.hist(Xcat[Xcat['Drivetrain'] == '4WD'].index, bins=30, alpha=0.65, label='4WD')
# plt.legend(loc='upper right')
# Xcat = df_objects.drop(['Country', 'Color', 'Upholstery'], axis=1).copy()
# Xcat = Xcat.drop(ind_drop, axis=0)
# Xcat.isna().sum()
# df_new = df.drop(ind_drop, axis=0)
# df_new.drop('Price', axis=1, inplace=True)
# df_new.select_dtypes(include='object').columns, Xcat.columns
# np.where(~df_new.columns.isin(Xcat.columns))[0]
# df_new.columns[3]
# for idx in np.where(~df_new.columns.isin(Xcat.columns))[0]:
#   col_name = df_new.columns[idx]
#   if col_name not in df_floats.columns.tolist() + Xcat.columns.tolist():

#     Xcat[col_name] = df_new[col_name]

# df_to_drop = pd.concat([Xcat.select_dtypes(include='number'), Xcat['Image_url']])
# (Xcat.drop(df_to_drop, axis=1).dtypes == 'object').all()

# Xcat.drop(df_to_drop.columns, axis=1, inplace=True)
# Xcat.columns

# Xcat.isna().sum()
# Xcat['Gearbox'].unique() #['Automatic', 'Manual', 'Semi-automatic', nan]
# Xcat['Gearbox'].hist(bins=30)
# plt.hist(Xcat[Xcat['Gearbox'] == 'Semi-automatic'].index, bins=7)
# Xcat['Gearbox'].isna().sum()
Xcat.isna().sum().sum() - 8634 # ['Fuel_Type','Gearbox','Model','Body',Seller] by dropping the nans in these columns we're dropping 881 rows from the dataset
#Xcat['Model_impu'] = Xcat.drop(np.where([Xcat['Model'].isna(), Xcat['Gearbox'].isna(), Xcat['Fuel_Type'].isna(),Xcat['Body'].isna(),Xcat['Seller'].isna()]))
#Xcat['Gearbox_impu']
#Xcat.drop(np.where([Xcat['Model'].isna(), Xcat['Gearbox'].isna()])[1])

# Xcat base has 40494 rows
# Xcat_new = Xcat.drop(np.concatenate((np.where([Xcat['Model'].isna(), Xcat['Gearbox'].isna(), Xcat['Fuel_Type'].isna()])[1], np.where([Xcat['Body'].isna(), Xcat['Seller'].isna()])[1])), axis=0)
drop1 = np.concatenate((np.where([Xcat['Model'].isna(), Xcat['Gearbox'].isna(), Xcat['Fuel_Type'].isna()])[1], np.where([Xcat['Body'].isna(), Xcat['Seller'].isna()])[1]))
# print(len(Xcat_new) - len(drop1))
# Xcat_new.isna().sum()
# Xcat.index.isin(Xcat_new.index).all() # np.False_

# len(set(Xcat.index).difference(set(Xcat_new.index))) #845 dropped

# Xcat_new.reset_index(drop=True, inplace=True)

# Xcat_new.isna().sum()

Xcat['Drivetrain_impu'] = Xcat['Drivetrain'].replace(np.nan,np.random.choice(['Front Wheel Drive', '4WD']) )
Xcat.drop('Drivetrain', axis=1, inplace=True)


In [ ]:

# Xcat_new.isna().sum()
# Xcat_new.columns.isin(Xcat.columns)
# Xcat_new = Xcat_new.astype('category')
# group = Xcat_new.groupby(['Make'])['Make']
# Xcat_new['Fuel_Type'].unique()



# for key, item in group:
#   print(group.get_group(key), "\n")


# We are now done with exploring and cleaning the dataset's object/categorical columns
# Now we will move on to cleaning/engineering the dataset's numeric features

/tmp/ipykernel_18639/3484275869.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group = Xcat_new.groupby(['Make'])['Make']


881

In [ ]:
df_floats.info()
df_floats['Price'] = df['Price']
df_floats['Engine_Size_cc'].mean()
df_floats['Engine_Size_cc'].describe()
df_floats[df_floats['Engine_Size_cc'] < 2500] # Near %60 of engine sizes are under 2500(mean), so we'll go with median

eng_size_median = df_floats['Engine_Size_cc'].median()
df_floats['Engine_Size_cc_impu'] = df_floats['Engine_Size_cc'].replace(np.nan, eng_size_median)




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40494 entries, 0 to 40493
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Year                 38799 non-null  float64
 1   Engine_Size_cc       34564 non-null  float64
 2   Seats                38697 non-null  float64
 3   Doors                39517 non-null  float64
 4   Price                40494 non-null  int64  
 5   Engine_Size_cc_impu  40494 non-null  float64
dtypes: float64(5), int64(1)
memory usage: 1.9 MB


In [ ]:
df_floats.isna().corr()

,Year,Engine_Size_cc,Seats,Doors,Price,Engine_Size_cc_impu
Year,1.000000,0.056427,0.011845,0.041873,NaN,NaN
Engine_Size_cc,0.056427,1.000000,0.250606,0.200259,NaN,NaN
Seats,0.011845,0.250606,1.000000,0.431119,NaN,NaN
Doors,0.041873,0.200259,0.431119,1.000000,NaN,NaN
Price,NaN,NaN,NaN,NaN,NaN,NaN
Engine_Size_cc_impu,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df_floats['Doors'].isna().sum())
print(df_floats['Seats'].isna().sum())
corr_drops = df_floats[df_floats['Doors'].isna() & df_floats['Seats'].isna() & df_floats['Engine_Size_cc'].isna()]['Year'].index.tolist()
len(corr_drops)



977
1797


428

In [ ]:
# Let's see if the distribution of the Year feature changes when we drop the correlated nan features of Doors, Seats, and Engine_Size_cc

df_floats["Year"].describe()
s_year = df_floats["Year"].copy()
s_year = s_year.drop(corr_drops,axis=0)

s_year.describe()

,Year
count,38398.000000
mean,2017.538856
std,8.892455
min,1922.000000
25%,2015.000000
50%,2020.000000
75%,2023.000000
max,2026.000000


In [ ]:
# Here we can see that the distribution for Year remains just about the same, so we'll drop the indexes for the correlated features
display(s_year.describe())
display(df_floats["Year"].describe())

In [ ]:
corr_drops_set = set(corr_drops)
drop1_set = set(drop1)
all_drops = corr_drops_set.update(drop1_set)
all_drops
df_floats.shape, Xcat.shape

((40494, 6), (40494, 8))

In [ ]:
corr_drops_set = set(corr_drops)
drop1_set = set(drop1)
all_drops = corr_drops_set.union(drop1_set)

Xnum = df_floats.drop(list(all_drops),axis=0)
year_median = Xnum.Year.median()
Xnum["Year_impu"] = Xnum["Year"].replace(np.nan, year_median)

Xnum = Xnum.drop(["Year", "Engine_Size_cc"],axis=1)
Xnum.isna().sum()
# Let's explore features Doors and Seats
Xnum['Doors'].describe()
Xnum['Seats'].describe()

Xnum['Doors'] = Xnum['Doors'].replace(np.nan, Xnum['Doors'].mean())
Xnum['Seats'] = Xnum['Seats'].replace(np.nan, Xnum['Seats'].mean())

Xcat = Xcat.drop(list(all_drops), axis=0)

Xnum.shape[0], Xcat.shape[0]

# #----------------

# Xcat_new.shape[0], Xnum.shape[0]
# drop = Xnum.shape[0] - Xcat_new.shape[0]
# print(drop)
# drop_choices = np.random.choice(Xnum.index, drop, replace=False)
# Xnum = Xnum.drop(drop_choices, axis=0)
# Xcat_new.shape[0], Xnum.shape[0]

# Now Xcat_new and Xnum have the same number of rows.
# They can now be concatenated to make the dataset that'll be used
# for train/test splits

(39401, 39401)

In [ ]:
fig, ax = plt.subplots(3, 1,figsize=(5,5))
i = 100


vc = Xcat['Model'].value_counts()
lt10 = Xcat['Model'].value_counts() > i

mods = [lt10.index[idx] for idx,val in enumerate(lt10) if val == True]
Xcat[Xcat['Model'].isin([idx for idx,val in enumerate(lt10) if val == True])].index





ax[0].plot(Xcat.index, Xcat.index, linewidth=0)
ax[0].scatter(Xcat[Xcat['Model'].isna()]['Model'].index, [0 for _ in range(Xcat[Xcat['Model'].isna()]['Model'].index.shape[0])], color='blue', clip_on=False, transform=ax[0].get_xaxis_transform())
ax[0].set_title("Spread of NA values for 'Model' Feature")

# ax[0].plot(Xcat.index, Xcat.index, linewidth=0)
# ax[0].scatter(Xcat[Xcat['Model'].isin(mods)].index, [0 for _ in range(Xcat[Xcat['Model'].isin(mods)].index.shape[0])], color='blue', clip_on=False, transform=ax[0].get_xaxis_transform())
# ax[0].set_title(f"Spread of 'Model' features with less than {i}")

z = 50
# while z <= 600:

#   lt10 = Xcat['Model'].value_counts() > z
#   mods = [lt10.index[idx] for idx,val in enumerate(lt10) if val == True]
plt.hist(Xcat[Xcat['Model'].isin(mods)].index, bins=30)
plt.xticks(rotation=90)
plt.show()

#   z += 50

plt.hist(Xcat[Xcat['Model'].isna()]['Model'].index, bins=30,edgecolor='black')
plt.xticks(rotation=90)
plt.show()
Xcat['Model'].value_counts().describe()


In [ ]:
Xcat['Model'].value_counts().sort_values()
Xcat[Xcat['Model'].isna()]
mods2 = Xcat['Model'][0:10000].value_counts()[0:10].index
Xcat[Xcat['Model'].isin(mods2)][2800:3250].head(50)
# mods2.drop('Leon')
#plt.hist(Xcat[Xcat['Model'].isin(mods2.drop(['Leon','500']))].index, bins=30, edgecolor='black')
#Xcat[Xcat['Model'].isna()]['Model'].reset_index()['index'].head(50)






In [ ]:
Xcat.isna().sum()
Xcat.shape[0], Xnum.shape[0]

y = Xnum['Price']

Xcat_new = Xcat.reset_index(drop=True)
Xnum_new = Xnum.reset_index(drop=True).drop('Price', axis=1)

Xcat_new.shape, Xnum_new.shape

Xnum_new
y.shape[0], Xcat_new.shape[0], Xnum_new.shape[0]
Xcat_new.nunique()




,0
Make,46
Model,1241
Body,9
Condition,2
Fuel_Type,10
Gearbox,3
Seller,2
Drivetrain_impu,3


In [ ]:
# one hot encode features Condition, Seller, and Gearbox
from sklearn.preprocessing import OneHotEncoder

def encoder_ohe(df, features):
  ohe = OneHotEncoder()

  for f in features:
    encoded_array = ohe.fit_transform(df[[f]]).toarray()
    df_encoded = pd.DataFrame(encoded_array, columns=[f"{cat}_encoded" for cat in ohe.categories_[0]])
    df = pd.concat([df, df_encoded],axis=1).drop(f, axis=1)
  return df

Xcat_temp = Xcat_new.copy()

features = ['Condition', 'Gearbox','Seller']
Xcat_encoded = encoder_ohe(Xcat_temp[features], features=['Condition', 'Gearbox','Seller'])
Xcat_encoded

X0 = pd.concat([Xcat_encoded, Xnum_new], axis=1)
X0.isna().sum()
X0.shape




(39401, 11)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Lasso, LassoCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(Xnum_new,y, test_size=0.2, random_state=42)

lasso_cv = LassoCV(alphas=[.0001, .001, .01, .1, .3, .4, .8, 1, 2, 3,10],random_state=0).fit(X_train, y_train)
dtr = DecisionTreeRegressor(max_depth = 7)
rfr = GradientBoostingRegressor(max_depth = 10, n_estimators=1000, random_state=42)
dtr.fit(X_train, y_train)
rfr.fit(X_train, y_train)

train_prediction = rfr.predict(X_train)
test_prediction = rfr.predict(X_test)

y_train_norm = (y_train - y_train.mean())/(y_train.std())
y_test_norm = (y_test - y_test.mean())/(y_test.std())

y_train_pred_norm = (train_prediction - train_prediction.mean())/(train_prediction.std())
y_test_pred_norm = (test_prediction - test_prediction.mean())/(test_prediction.std())

mse_train = mean_squared_error(y_train, train_prediction)
mse_test = mean_squared_error(y_test, test_prediction)

mse_train, mse_test

mse_train**(1/2), mse_test**(1/2)




(8603193948.948175, 12430699374.462591)

In [ ]:
8603193948.948175**(1/2)

92753.40397499262

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(learning_rate=0.1, n_estimators=100)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_pred_xgb_norm = (y_pred_xgb - y_pred_xgb.mean())/(y_pred_xgb.std())
xgb_df = pd.DataFrame({"Actual":y_test, "Prediction":y_pred_xgb})
((xgb_df['Actual'] - xgb_df['Prediction'])).describe()

,0
count,7.881000e+03
mean,-1.622497e+03
std,1.086387e+05
min,-1.438361e+06
25%,-7.204391e+03
50%,-1.717414e+03
75%,2.991377e+03
max,4.083188e+06


In [ ]:
import joblib
joblib.dump(xgb, "used_car_best_model_xgb.pkl")

['used_car_best_model_xgb.pkl']

In [ ]:
X_train.columns.tolist()

['Seats', 'Doors', 'Engine_Size_cc_impu', 'Year_impu']